<a href="https://colab.research.google.com/github/RuleOfNight/phishing-detector/blob/main/98_55_20(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -----------------------------------------------------------------------------
# CÀI ĐẶT THƯ VIỆN (MINIMAL INSTALLATION)
# -----------------------------------------------------------------------------
!pip install -q transformers peft pandas spacy
!python -m spacy download en_core_web_sm -q

import os
import torch
import pandas as pd
import spacy
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 123.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
df = pd.read_csv("phishing_email.csv")
df.head()

,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [ ]:
# Kiểm tra GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Hardware: {device}")

# Kiểm tra file dữ liệu
if not os.path.exists('phishing_email.csv'):
    raise FileNotFoundError("❌ LỖI: Vui lòng upload file 'phishing_email.csv' lên Colab trước khi chạy!")
else:
    print("✅ Đã tìm thấy file dữ liệu.")

✅ Hardware: cuda
✅ Đã tìm thấy file dữ liệu.


In [ ]:
# -----------------------------------------------------------------------------
# CÀI ĐẶT THƯ VIỆN & CHUẨN BỊ
# -----------------------------------------------------------------------------
!pip install -q transformers peft pandas spacy
!python -m spacy download en_core_web_sm -q

import os
import torch
import pandas as pd
import spacy
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# Kiểm tra GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Hardware: {device}")

# -----------------------------------------------------------------------------
# DATASET CLASS (ĐÃ CẬP NHẬT CHO FORMAT MỚI)
# -----------------------------------------------------------------------------
class HeteroGraphDataset(Dataset):
    def __init__(self, csv_file, tokenizer_name='microsoft/deberta-v3-base', max_len=256):
        print("⏳ Đang đọc dữ liệu và xử lý nhãn...")

        # 1. Đọc dữ liệu
        # Sử dụng try-except để bắt lỗi nếu file không đúng định dạng
        try:
            self.data = pd.read_csv(csv_file)
        except Exception as e:
            print(f"❌ Lỗi đọc file csv: {e}")
            raise

        # 2. Xử lý & Ánh xạ nhãn (Label Mapping)
        # Chuyển đổi từ String ('Safe Email') sang Int (0, 1)
        label_map = {
            'Safe Email': 0,
            'Phishing Email': 1
        }

        # Kiểm tra xem cột 'Email Type' có tồn tại không
        if 'Email Type' not in self.data.columns or 'Email Text' not in self.data.columns:
            raise ValueError(f"❌ File CSV phải chứa cột 'Email Text' và 'Email Type'. Các cột hiện có: {list(self.data.columns)}")

        # Map dữ liệu
        self.data['label_idx'] = self.data['Email Type'].map(label_map)

        # Lọc bỏ các dòng bị lỗi (NaN) ở Text hoặc Label
        before_drop = len(self.data)
        self.data = self.data.dropna(subset=['Email Text', 'label_idx']).reset_index(drop=True)
        print(f"🧹 Đã lọc bỏ {before_drop - len(self.data)} dòng lỗi/trống.")

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.max_len = max_len

        # Load Spacy (POS Tagger)
        self.nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])

        # Tạo từ điển POS
        self.pos_labels = self.nlp.get_pipe("tagger").labels
        self.pos2id = {label: i+1 for i, label in enumerate(self.pos_labels)}
        self.pos2id['UNK'] = len(self.pos2id) + 1
        self.pos2id['PAD'] = 0

        print(f"✅ Đã load xong. Tổng số mẫu sạch: {len(self.data)}")
        print(f"✅ Cấu hình POS Vocab Size: {len(self.pos2id)}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # 1. Lấy dữ liệu (Theo tên cột mới)
        text = str(self.data.iloc[idx]['Email Text'])     # Cột 'Email Text'
        label = int(self.data.iloc[idx]['label_idx'])     # Cột đã map sang số

        # 2. Phân tích ngữ pháp (Cắt ngắn để tránh OOM Spacy)
        doc = self.nlp(text[:1500])

        # 3. Sub-word Alignment
        tokens = []
        pos_ids = []

        for token in doc:
            sub_tokens = self.tokenizer.tokenize(token.text)
            p_id = self.pos2id.get(token.tag_, self.pos2id['UNK'])
            for sub_token in sub_tokens:
                tokens.append(sub_token)
                pos_ids.append(p_id)

        # 4. Truncate & Special Tokens
        if len(tokens) > self.max_len - 2:
            tokens = tokens[:self.max_len - 2]
            pos_ids = pos_ids[:self.max_len - 2]

        input_ids = self.tokenizer.convert_tokens_to_ids(tokens)
        input_ids = [self.tokenizer.cls_token_id] + input_ids + [self.tokenizer.sep_token_id]
        pos_ids = [0] + pos_ids + [0]

        # 5. Padding
        attention_mask = [1] * len(input_ids)
        padding_len = self.max_len - len(input_ids)

        input_ids = input_ids + [self.tokenizer.pad_token_id] * padding_len
        pos_ids = pos_ids + [0] * padding_len
        attention_mask = attention_mask + [0] * padding_len

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'pos_ids': torch.tensor(pos_ids, dtype=torch.long),
            'label': torch.tensor(label, dtype=torch.long)
        }

# -----------------------------------------------------------------------------
# KIỂM TRA DỮ LIỆU
# -----------------------------------------------------------------------------
BATCH_SIZE = 16
print("🚀 Đang khởi tạo Dataset (Phiên bản mới)...")

# Lưu ý: Đảm bảo file csv tên đúng là file bạn đã upload (ví dụ: 'phishing_data.csv' hoặc tên cũ)
# Ở đây tôi vẫn để tên cũ, bạn hãy đổi tên file trong lệnh dưới nếu file bạn up lên có tên khác
dataset = HeteroGraphDataset('phishing_email.csv')

dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

sample_batch = next(iter(dataloader))
print("\n🔍 KIỂM TRA FORMAT MỚI:")
print(f" - Input IDs: {sample_batch['input_ids'].shape}")
print(f" - Labels (Example): {sample_batch['label'][:5]}") # In ra vài nhãn để xem có đúng 0/1 không
print("✅ Bước 1 (Sửa đổi) Hoàn tất.")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ Hardware: cuda
🚀 Đang khởi tạo Dataset (Phiên bản mới)...
⏳ Đang đọc dữ liệu và xử lý nhãn...
🧹 Đã lọc bỏ 16 dòng lỗi/trống.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

✅ Đã load xong. Tổng số mẫu sạch: 18634
✅ Cấu hình POS Vocab Size: 52

🔍 KIỂM TRA FORMAT MỚI:
 - Input IDs: torch.Size([16, 256])
 - Labels (Example): tensor([0, 1, 1, 0, 0])
✅ Bước 1 (Sửa đổi) Hoàn tất.


In [ ]:
# Cấu hình batch_size phù hợp với GPU Colab (T4 thường chịu được 16 hoặc 32)
BATCH_SIZE = 16

print("🚀 Đang khởi tạo Dataset thực tế (Có thể mất 1-2 phút để load spacy lần đầu)...")
dataset = HeteroGraphDataset('phishing_email.csv')
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Kiểm tra nhanh 1 batch để chắc chắn mọi thứ đúng kích thước Blueprint
sample_batch = next(iter(dataloader))
print("\n🔍 KIỂM TRA KÍCH THƯỚC TENSOR (Blueprint Check):")
print(f" - Input IDs (Text Branch): {sample_batch['input_ids'].shape} | Kỳ vọng: [{BATCH_SIZE}, 256]")
print(f" - POS IDs (Syntax Branch): {sample_batch['pos_ids'].shape}   | Kỳ vọng: [{BATCH_SIZE}, 256]")
print(f" - Labels: {sample_batch['label'].shape}")
print("✅ Bước 1 Hoàn tất. Dữ liệu đã sẵn sàng cho Architecture.")

🚀 Đang khởi tạo Dataset thực tế (Có thể mất 1-2 phút để load spacy lần đầu)...
⏳ Đang đọc dữ liệu và xử lý nhãn...
🧹 Đã lọc bỏ 16 dòng lỗi/trống.
✅ Đã load xong. Tổng số mẫu sạch: 18634
✅ Cấu hình POS Vocab Size: 52

🔍 KIỂM TRA KÍCH THƯỚC TENSOR (Blueprint Check):
 - Input IDs (Text Branch): torch.Size([16, 256]) | Kỳ vọng: [16, 256]
 - POS IDs (Syntax Branch): torch.Size([16, 256])   | Kỳ vọng: [16, 256]
 - Labels: torch.Size([16])
✅ Bước 1 Hoàn tất. Dữ liệu đã sẵn sàng cho Architecture.


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoConfig
from peft import get_peft_model, LoraConfig, TaskType

# -----------------------------------------------------------------------------
# 1. MODALITY ADAPTER (MODULE TÁI SỬ DỤNG)
# -----------------------------------------------------------------------------
class ModalityAdapter(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.15):
        super().__init__()
        self.projector = nn.Sequential(
            nn.Linear(input_dim, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.projector(x)

# -----------------------------------------------------------------------------
# 2. DUAL-BRANCH ENCODER (HETEROGENEOUS BACKBONE)
# -----------------------------------------------------------------------------
class DualBranchEncoder(nn.Module):
    def __init__(self,
                 model_name='microsoft/deberta-v3-base',
                 pos_vocab_size=55, # Lấy từ kết quả Step 1 (52 + dư vài slot)
                 pos_embed_dim=128,
                 d_model=256):
        super().__init__()

        # --- NHÁNH 1: TEXT BRANCH (DeBERTa + LoRA) ---
        print(f"🔧 Đang tải {model_name} và cài đặt LoRA...")
        self.deberta = AutoModel.from_pretrained(model_name)

        # Cấu hình LoRA (Chỉ train adapter, đóng băng backbone)
        peft_config = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            inference_mode=False,
            r=16,            # Rank nhỏ để tiết kiệm RAM
            lora_alpha=32,
            lora_dropout=0.1,
            target_modules=["query_proj", "value_proj"] # Nhắm vào Q, V của Attention
        )
        self.deberta = get_peft_model(self.deberta, peft_config)
        self.text_dim = 768 # Kích thước gốc của DeBERTa Base

        # --- NHÁNH 2: SYNTAX BRANCH (POS Embedding) ---
        self.pos_embedding = nn.Embedding(pos_vocab_size, pos_embed_dim, padding_idx=0)

        # --- MODALITY ADAPTERS ---
        # Adapter cho Text: 768 -> 256
        self.text_adapter = ModalityAdapter(self.text_dim, d_model)
        # Adapter cho POS: 50 -> 256
        self.pos_adapter = ModalityAdapter(pos_embed_dim, d_model)

        # --- TYPE EMBEDDING ---
        # 0: Text Type, 1: POS Type
        self.type_embedding = nn.Embedding(2, d_model)

    def forward(self, input_ids, attention_mask, pos_ids):
        # 1. Xử lý Text Branch
        deberta_out = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        h_text_raw = deberta_out.last_hidden_state

        # 🟢 THÊM DÒNG NÀY ĐỂ FIX LỖI: Ép kiểu output của DeBERTa về float32
        h_text_raw = h_text_raw.to(torch.float32)

        # 2. Xử lý Syntax Branch
        h_pos_raw = self.pos_embedding(pos_ids)
        # Ép kiểu luôn pos_raw cho chắc chắn (dù thường nó đã là float32)
        h_pos_raw = h_pos_raw.to(torch.float32)

        # 3. Qua Adapters (Chiếu về 256)
        h_text_proj = self.text_adapter(h_text_raw)
        h_pos_proj = self.pos_adapter(h_pos_raw)

        # 4. Cộng Type Embedding (Đóng dấu bản chất)
        batch_size, seq_len = input_ids.shape
        device = input_ids.device

        type_text_idx = torch.zeros((batch_size, seq_len), dtype=torch.long, device=device)
        type_pos_idx = torch.ones((batch_size, seq_len), dtype=torch.long, device=device)

        h_text_final = h_text_proj + self.type_embedding(type_text_idx)
        h_pos_final = h_pos_proj + self.type_embedding(type_pos_idx)

        return h_text_final, h_pos_final

# -----------------------------------------------------------------------------
# 3. KIỂM TRA NGAY TRÊN BATCH DỮ LIỆU THỰC
# -----------------------------------------------------------------------------
print("🚀 Đang khởi tạo DualBranchEncoder (Có LoRA)...")
# Lấy vocab size từ dataset ở Step 1 (dataset.pos2id bao gồm cả UNK và PAD)
POS_VOCAB_SIZE = len(dataset.pos2id) + 1

encoder_model = DualBranchEncoder(pos_vocab_size=POS_VOCAB_SIZE).to(device)

# Lấy 1 batch thực tế từ Dataloader Step 1
batch = next(iter(dataloader))
ids = batch['input_ids'].to(device)
mask = batch['attention_mask'].to(device)
pos = batch['pos_ids'].to(device)

# Forward pass
h_text, h_pos = encoder_model(ids, mask, pos)

print("\n🔍 KIỂM TRA OUTPUT BƯỚC 2 (Blueprint Check):")
print(f" - H_Text (Projected + Type): {h_text.shape} | Kỳ vọng: [16, 256, 256]")
print(f" - H_POS  (Projected + Type): {h_pos.shape}  | Kỳ vọng: [16, 256, 256]")
print(f" - LoRA Trainable Parameters: {encoder_model.deberta.print_trainable_parameters()}")
print("✅ Bước 2 Hoàn tất. Hai nhánh đặc trưng đã sẵn sàng cho Graph Transformer.")

🚀 Đang khởi tạo DualBranchEncoder (Có LoRA)...
🔧 Đang tải microsoft/deberta-v3-base và cài đặt LoRA...


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]


🔍 KIỂM TRA OUTPUT BƯỚC 2 (Blueprint Check):
 - H_Text (Projected + Type): torch.Size([16, 256, 256]) | Kỳ vọng: [16, 256, 256]
 - H_POS  (Projected + Type): torch.Size([16, 256, 256])  | Kỳ vọng: [16, 256, 256]
trainable params: 589,824 || all params: 184,421,376 || trainable%: 0.3198
 - LoRA Trainable Parameters: None
✅ Bước 2 Hoàn tất. Hai nhánh đặc trưng đã sẵn sàng cho Graph Transformer.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# -----------------------------------------------------------------------------
# 1. GRAPH EDGE BIAS GENERATOR (TẠO LUẬT KẾT NỐI)
# -----------------------------------------------------------------------------
class SmartHeteroEdgeBias(nn.Module):
    def __init__(self, num_heads, max_seq_len=256, max_rel_dist=16):
        super().__init__()
        self.num_heads = num_heads
        self.max_dist = max_rel_dist

        # 1. Base Type Bias: Học xu hướng kết nối chung giữa 4 vùng (T-T, T-P, P-T, P-P)
        self.type_bias = nn.Parameter(torch.randn(num_heads, 2, 2) * 0.02)

        # 2. Learnable Alignment Weights (Thay cho số 1.0 cứng)
        # Mỗi Head tự quyết định độ mạnh của đường chéo chính (i == j)
        self.align_weight_main = nn.Parameter(torch.ones(num_heads, 1, 1) * 0.5)
        # Độ mạnh của các node lân cận (i == j-1 hoặc i == j+1) để chống nhiễu sub-word
        self.align_weight_sub = nn.Parameter(torch.ones(num_heads, 1, 1) * 0.1)

        # 3. Relative Positional Bias (Cho T-T và P-P)
        # Nhúng khoảng cách tương đối giữa các node (từ -max_dist đến +max_dist)
        self.rel_pos_embedding = nn.Embedding(2 * self.max_dist + 1, num_heads)

        # Pre-compute ma trận khoảng cách tương đối (để chạy nhanh hơn)
        positions = torch.arange(max_seq_len)
        rel_pos = positions.unsqueeze(1) - positions.unsqueeze(0) # [L, L]
        # Giới hạn khoảng cách và dịch trần về số dương
        rel_pos = torch.clamp(rel_pos, -self.max_dist, self.max_dist) + self.max_dist
        self.register_buffer('rel_pos_idx', rel_pos)

    def forward(self, seq_len):
        device = self.type_bias.device
        total_len = 2 * seq_len
        bias = torch.zeros(self.num_heads, total_len, total_len, device=device)

        # --- PHẦN 1: GLOBAL TYPE BIAS ---
        bias[:, :seq_len, :seq_len] += self.type_bias[:, 0, 0].view(-1, 1, 1)
        bias[:, :seq_len, seq_len:] += self.type_bias[:, 0, 1].view(-1, 1, 1)
        bias[:, seq_len:, :seq_len] += self.type_bias[:, 1, 0].view(-1, 1, 1)
        bias[:, seq_len:, seq_len:] += self.type_bias[:, 1, 1].view(-1, 1, 1)

        # --- PHẦN 2: SMART LOCAL ALIGNMENT (TEXT <-> POS) ---
        # Tạo ma trận đường chéo chính (Main band)
        eye_main = torch.eye(seq_len, device=device).unsqueeze(0) # [1, L, L]

        # Tạo ma trận đường chéo phụ (Sub band: bù đắp lệch pha tokenization)
        idx = torch.arange(seq_len, device=device)
        dist_matrix = torch.abs(idx.unsqueeze(1) - idx.unsqueeze(0))
        eye_sub = (dist_matrix == 1).float().unsqueeze(0) # [1, L, L]

        # Tính tổng Bias căn chỉnh có trọng số học được
        cross_align = (eye_main * self.align_weight_main) + (eye_sub * self.align_weight_sub)

        # Tiêm vào vùng Text-POS và POS-Text
        bias[:, :seq_len, seq_len:] += cross_align
        bias[:, seq_len:, :seq_len] += cross_align

        # --- PHẦN 3: RELATIVE POSITIONAL BIAS (TEXT-TEXT & POS-POS) ---
        # Lấy khoảng cách tương ứng với seq_len hiện tại
        rp_idx = self.rel_pos_idx[:seq_len, :seq_len] # [L, L]
        rp_bias = self.rel_pos_embedding(rp_idx).permute(2, 0, 1) # [Heads, L, L]

        # Cộng vào nội bộ từng Modality
        bias[:, :seq_len, :seq_len] += rp_bias
        bias[:, seq_len:, seq_len:] += rp_bias

        return bias

# -----------------------------------------------------------------------------
# 2. HETEROGENEOUS GRAPH TRANSFORMER LAYER (CORE)
# -----------------------------------------------------------------------------
class GraphTransformerLayer(nn.Module):
    def __init__(self, hidden_dim=256, num_heads=8, dropout=0.15):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.scale = self.head_dim ** -0.5

        # Q, K, V Projections
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.o_proj = nn.Linear(hidden_dim, hidden_dim)

        # Edge Bias Generator
        self.edge_bias_generator = SmartHeteroEdgeBias(num_heads=num_heads, max_seq_len=256)

        # Norm & FeedForward
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 4, hidden_dim)
        )

    def forward(self, h_text, h_pos, attention_mask=None):
        """
        Input:
            h_text: [Batch, Seq, Dim]
            h_pos:  [Batch, Seq, Dim]
            mask:   [Batch, Seq] (Mask gốc của Text)
        """
        batch_size, seq_len, _ = h_text.shape

        # 1. FUSION (UNION OF NODES) - Ghép nối node vào đồ thị chung
        # GraphNodes: [TextNodes, POSNodes] -> Shape: [Batch, 2*Seq, Dim]
        h_graph = torch.cat([h_text, h_pos], dim=1)

        # 2. Tạo Extended Mask cho cả Graph
        # Mask gốc chỉ có Text, ta cần mask cho cả phần POS (POS cũng dùng mask của Text vì 1-1)
        # GraphMask: [Batch, 2*Seq]
        if attention_mask is not None:
            graph_mask = torch.cat([attention_mask, attention_mask], dim=1)
            # Chuyển đổi sang dạng cộng (-inf cho padding)
            # [Batch, 1, 1, 2*Seq]
            extended_mask = (1.0 - graph_mask.unsqueeze(1).unsqueeze(2)) * -1e9
        else:
            extended_mask = 0.0

        # 3. Self-Attention Logic
        residual = h_graph
        x = self.norm1(h_graph)

        # Chiếu Q, K, V
        Q = self.q_proj(x).view(batch_size, 2*seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(batch_size, 2*seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(batch_size, 2*seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Tính Score cơ bản: Q * K^T
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        # 4. INJECT EDGE BIAS (Heterogeneous Rules)
        # Lấy bias [Heads, 2*Seq, 2*Seq]
        edge_bias = self.edge_bias_generator(seq_len)
        attn_scores = attn_scores + edge_bias.unsqueeze(0) # Broadcast batch

        # Cộng Mask padding
        attn_scores = attn_scores + extended_mask

        # Softmax & Context
        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_probs = self.dropout(attn_probs)

        context = torch.matmul(attn_probs, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, 2*seq_len, -1)
        output = self.o_proj(context)

        # Residual 1
        x = residual + self.dropout(output)

        # FFN & Residual 2
        x = x + self.ffn(self.norm2(x))

        # 5. SPLIT OUTPUT (Tách lại về 2 luồng nếu cần dùng cho layer sau hoặc pooling)
        h_text_out = x[:, :seq_len, :]
        h_pos_out = x[:, seq_len:, :]

        return h_text_out, h_pos_out

# -----------------------------------------------------------------------------
# 3. KIỂM TRA BƯỚC 3 (Blueprint Check)
# -----------------------------------------------------------------------------
print("🚀 Đang khởi tạo Graph Transformer Layer...")
graph_layer = GraphTransformerLayer(hidden_dim=256, num_heads=4).to(device)

# Lấy input giả lập từ kết quả Step 2 (để test logic layer này độc lập)
# h_text, h_pos có shape [16, 256, 256] và mask [16, 256]
with torch.no_grad():
    out_text, out_pos = graph_layer(h_text, h_pos, mask)

print("\n🔍 KIỂM TRA OUTPUT BƯỚC 3:")
print(f" - Graph Input Nodes: {h_text.shape[1] + h_pos.shape[1]} (256 Text + 256 POS)")
print(f" - Output Text Node:  {out_text.shape} | Kỳ vọng: [16, 256, 256]")
print(f" - Output POS Node:   {out_pos.shape}  | Kỳ vọng: [16, 256, 256]")
print("✅ Bước 3 Hoàn tất. Lõi Graph đã hoạt động với Edge Bias.")

🚀 Đang khởi tạo Graph Transformer Layer...

🔍 KIỂM TRA OUTPUT BƯỚC 3:
 - Graph Input Nodes: 512 (256 Text + 256 POS)
 - Output Text Node:  torch.Size([16, 256, 256]) | Kỳ vọng: [16, 256, 256]
 - Output POS Node:   torch.Size([16, 256, 256])  | Kỳ vọng: [16, 256, 256]
✅ Bước 3 Hoàn tất. Lõi Graph đã hoạt động với Edge Bias.


In [ ]:
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tqdm.auto import tqdm
import numpy as np # Đảm bảo import numpy

# -----------------------------------------------------------------------------
# 1. FULL MODEL ASSEMBLY (GIỮ NGUYÊN)
# -----------------------------------------------------------------------------
class PhishingHeteroGraphModel(nn.Module):
    def __init__(self, encoder, graph_layers=2, hidden_dim=256, num_classes=2):
        super().__init__()
        self.encoder = encoder # DualBranchEncoder từ Step 2

        # Stack nhiều lớp Graph Transformer
        self.gps_layers = nn.ModuleList([
            GraphTransformerLayer(hidden_dim=hidden_dim, num_heads=4)
            for _ in range(graph_layers)
        ])

        # Classifier Head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, input_ids, attention_mask, pos_ids):
        # 1. Feature Extraction
        h_text, h_pos = self.encoder(input_ids, attention_mask, pos_ids)

        # 2. Graph Reasoning
        for layer in self.gps_layers:
            h_text, h_pos = layer(h_text, h_pos, attention_mask)

        # 3. Global Pooling (Lấy [CLS] của nhánh Text)
        cls_embedding = h_text[:, 0, :]

        # 4. Classification
        logits = self.classifier(cls_embedding)
        return logits

# -----------------------------------------------------------------------------
# 2. CHUẨN BỊ DỮ LIỆU (TRAIN/VAL/TEST SPLIT - 80/10/10)
# -----------------------------------------------------------------------------
print("✂️ Đang chia dữ liệu theo tỷ lệ 80/10/10...")

# Lấy toàn bộ index
full_indices = np.arange(len(dataset))

# Lần cắt 1: Tách Train (80%) và Temp (20%)
train_idx, temp_idx = train_test_split(full_indices, test_size=0.2, random_state=42)

# Lần cắt 2: Tách Temp (20%) thành Val (10%) và Test (10%)
# Vì Temp đang là 20% tổng, nên cắt đôi (0.5) sẽ được mỗi bên 10% tổng
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

# Tạo Subset
train_set = torch.utils.data.Subset(dataset, train_idx)
val_set   = torch.utils.data.Subset(dataset, val_idx)
test_set  = torch.utils.data.Subset(dataset, test_idx)

# Tạo Loader
# Train cần shuffle để học tốt hơn, Val/Test không cần shuffle
train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_set, batch_size=16, shuffle=False)
test_loader  = DataLoader(test_set, batch_size=16, shuffle=False)

print(f"✅ Tổng số mẫu: {len(dataset)}")
print(f"   - Train Set (80%): {len(train_set)} mẫu")
print(f"   - Val Set   (10%): {len(val_set)} mẫu")
print(f"   - Test Set  (10%): {len(test_set)} mẫu")

# -----------------------------------------------------------------------------
# 3. HÀM QUẢN LÝ (TRAIN & EVALUATE)
# -----------------------------------------------------------------------------
def freeze_backbone(model):
    print("\n🔒 MODE: WARM-UP (Đóng băng DeBERTa & LoRA)")
    for param in model.parameters():
        param.requires_grad = False

    trainable_modules = [
        model.encoder.pos_embedding,
        model.encoder.text_adapter,
        model.encoder.pos_adapter,
        model.encoder.type_embedding,
        model.gps_layers,
        model.classifier
    ]
    for module in trainable_modules:
        for param in module.parameters():
            param.requires_grad = True
    print(f"👉 Ready for Phase 1.")

def unfreeze_lora(model):
    print("\n🔓 MODE: JOINT TRAINING (Mở khóa LoRA)")
    for name, param in model.encoder.deberta.named_parameters():
        if "lora_" in name:
            param.requires_grad = True
    print(f"👉 Ready for Phase 2.")

# Hàm Train
def train_engine(model, dataloader, optimizer, criterion, phase_name):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(dataloader, desc=f"Training {phase_name}")

    for batch in progress_bar:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        pos = batch['pos_ids'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(ids, mask, pos)
        loss = criterion(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        progress_bar.set_postfix({'loss': loss.item(), 'acc': correct/total})

    return total_loss / len(dataloader), correct / total

# Hàm Evaluate (Dùng cho Val và Test)
def evaluate_engine(model, dataloader, criterion, mode="Val"):
    model.eval() # Chế độ đánh giá (tắt Dropout)
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad(): # Không tính gradient
        for batch in tqdm(dataloader, desc=f"Evaluating ({mode})"):
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            pos = batch['pos_ids'].to(device)
            labels = batch['label'].to(device)

            logits = model(ids, mask, pos)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    acc = correct / total
    print(f"   📊 {mode} Result: Loss = {avg_loss:.4f} | Accuracy = {acc:.4f}")
    return avg_loss, acc

# -----------------------------------------------------------------------------
# 4. THỰC THI (EXECUTION)
# -----------------------------------------------------------------------------
full_model = PhishingHeteroGraphModel(encoder_model, graph_layers=1).to(device)
criterion = nn.CrossEntropyLoss()

# === PHASE 1: WARM-UP ===
freeze_backbone(full_model)
optimizer_warmup = optim.AdamW(filter(lambda p: p.requires_grad, full_model.parameters()), lr=1e-3)

print("\n🚀 BẮT ĐẦU GIAI ĐOẠN 1: WARM-UP GNN...")
train_loss, train_acc = train_engine(full_model, train_loader, optimizer_warmup, criterion, "Phase 1")
# Đánh giá ngay trên tập Val sau khi Warmup
val_loss, val_acc = evaluate_engine(full_model, val_loader, criterion, mode="Val")

print("\n🚀 BẮT ĐẦU GIAI ĐOẠN 1: WARM-UP GNN..._Epochs2")
train_loss, train_acc = train_engine(full_model, train_loader, optimizer_warmup, criterion, "Phase 1")
# Đánh giá ngay trên tập Val sau khi Warmup
val_loss, val_acc = evaluate_engine(full_model, val_loader, criterion, mode="Val")

# === PHASE 2: JOINT TRAINING ===
unfreeze_lora(full_model)
param_groups = [
    {'params': [p for n, p in full_model.named_parameters() if "lora_" in n], 'lr': 5e-3},
    {'params': [p for n, p in full_model.named_parameters() if "lora_" not in n and p.requires_grad], 'lr': 5e-4}
]
optimizer_joint = optim.AdamW(param_groups)

print("\n🚀 BẮT ĐẦU GIAI ĐOẠN 2: JOINT TRAINING (LoRA + GNN)...")
train_loss, train_acc = train_engine(full_model, train_loader, optimizer_joint, criterion, "Phase 2")

# Đánh giá trên Val
print("--- Đánh giá mô hình trên tập Validation (Dùng để chỉnh sửa) ---")
val_loss, val_acc = evaluate_engine(full_model, val_loader, criterion, mode="Val")

# Đánh giá trên Test (Chỉ chạy 1 lần cuối cùng)
print("\n--- Đánh giá mô hình trên tập Test (Kết quả cuối cùng) ---")
test_loss, test_acc = evaluate_engine(full_model, test_loader, criterion, mode="Test")

print(f"\n🎉 KẾT QUẢ CUỐI CÙNG TRÊN TẬP TEST: {test_acc:.2%}")

✂️ Đang chia dữ liệu theo tỷ lệ 80/10/10...
✅ Tổng số mẫu: 18634
   - Train Set (80%): 14907 mẫu
   - Val Set   (10%): 1863 mẫu
   - Test Set  (10%): 1864 mẫu

🔒 MODE: WARM-UP (Đóng băng DeBERTa & LoRA)
👉 Ready for Phase 1.

🚀 BẮT ĐẦU GIAI ĐOẠN 1: WARM-UP GNN...


Training Phase 1:   0%|          | 0/932 [00:00<?, ?it/s]

Evaluating (Val):   0%|          | 0/117 [00:00<?, ?it/s]

   📊 Val Result: Loss = 0.1835 | Accuracy = 0.9501

🚀 BẮT ĐẦU GIAI ĐOẠN 1: WARM-UP GNN..._Epochs2


Training Phase 1:   0%|          | 0/932 [00:00<?, ?it/s]

Evaluating (Val):   0%|          | 0/117 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:

# === PHASE 2: JOINT TRAINING ===
unfreeze_lora(full_model)
param_groups = [
    {'params': [p for n, p in full_model.named_parameters() if "lora_" in n], 'lr': 3e-4},
    {'params': [p for n, p in full_model.named_parameters() if "lora_" not in n and p.requires_grad], 'lr': 5e-4}
]
optimizer_joint = optim.AdamW(param_groups)

print("\n🚀 BẮT ĐẦU GIAI ĐOẠN 2: JOINT TRAINING (LoRA + GNN)...")
train_loss, train_acc = train_engine(full_model, train_loader, optimizer_joint, criterion, "Phase 2")

# Đánh giá trên Val
print("--- Đánh giá mô hình trên tập Validation (Dùng để chỉnh sửa) ---")
val_loss, val_acc = evaluate_engine(full_model, val_loader, criterion, mode="Val")

# Đánh giá trên Test (Chỉ chạy 1 lần cuối cùng)
print("\n--- Đánh giá mô hình trên tập Test (Kết quả cuối cùng) ---")
test_loss, test_acc = evaluate_engine(full_model, test_loader, criterion, mode="Test")

print(f"\n🎉 KẾT QUẢ CUỐI CÙNG TRÊN TẬP TEST: {test_acc:.2%}")


🔓 MODE: JOINT TRAINING (Mở khóa LoRA)
👉 Ready for Phase 2.

🚀 BẮT ĐẦU GIAI ĐOẠN 2: JOINT TRAINING (LoRA + GNN)...


Training Phase 2:   0%|          | 0/932 [00:00<?, ?it/s]

--- Đánh giá mô hình trên tập Validation (Dùng để chỉnh sửa) ---


Evaluating (Val):   0%|          | 0/117 [00:00<?, ?it/s]

   📊 Val Result: Loss = 0.0940 | Accuracy = 0.9705

--- Đánh giá mô hình trên tập Test (Kết quả cuối cùng) ---


Evaluating (Test):   0%|          | 0/117 [00:00<?, ?it/s]

   📊 Test Result: Loss = 0.0472 | Accuracy = 0.9823

🎉 KẾT QUẢ CUỐI CÙNG TRÊN TẬP TEST: 98.23%


In [ ]:
best_val_acc = 0.0
val_loss, val_acc = evaluate_engine(
    full_model,
    val_loader,
    criterion,
    mode=f"Val (Epoch {epoch + 1})"
)

# 3. Cơ chế Lưu mô hình tốt nhất (Early Saving)
if val_acc > best_val_acc:
    best_val_acc = val_acc
    torch.save(full_model.state_dict(), "best_phishing_model.pt")
    print(f"🌟 Đã lưu mô hình có Val Acc tốt nhất: {val_acc:.4f}")


Evaluating (Val (Epoch 1)):   0%|          | 0/117 [00:00<?, ?it/s]

   📊 Val (Epoch 1) Result: Loss = 0.1017 | Accuracy = 0.9742
🌟 Đã lưu mô hình có Val Acc tốt nhất: 0.9742


In [ ]:
# -----------------------------------------------------------------------------
# 5. TIẾP TỤC HUẤN LUYỆN BỔ SUNG (5 EPOCHS)
# -----------------------------------------------------------------------------
num_epochs = 2
unfreeze_lora(full_model)
optimizer_warmup = optim.AdamW(filter(lambda p: p.requires_grad, full_model.parameters()), lr=5e-4)
print(f"\n🔥 BẮT ĐẦU HUẤN LUYỆN BỔ SUNG {num_epochs} EPOCHS (JOINT TRAINING)...")

for epoch in range(num_epochs):
    print(f"\n--- Epoch {epoch + 1}/{num_epochs} ---")

    # 1. Chạy huấn luyện trên tập Train
    # Sử dụng optimizer_joint để cập nhật cả LoRA và các lớp GNN
    train_loss, train_acc = train_engine(
        full_model,
        train_loader,
        optimizer_joint,
        criterion,
        f"Epoch {epoch + 1}"
    )

    # 2. Đánh giá trên tập Validation sau mỗi epoch
    val_loss, val_acc = evaluate_engine(
        full_model,
        val_loader,
        criterion,
        mode=f"Val (Epoch {epoch + 1})"
    )

    # 3. Cơ chế Lưu mô hình tốt nhất (Early Saving)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(full_model.state_dict(), "best_phishing_model.pt")
        print(f"🌟 Đã lưu mô hình có Val Acc tốt nhất: {val_acc:.4f}")

# -----------------------------------------------------------------------------
# 6. ĐÁNH GIÁ CUỐI CÙNG SAU KHI TRAIN XONG
# -----------------------------------------------------------------------------
print("\n--- Đánh giá cuối cùng với mô hình tốt nhất vừa lưu ---")
# Tải lại trọng số tốt nhất trước khi test
full_model.load_state_dict(torch.load("best_phishing_model.pt"))

test_loss, test_acc = evaluate_engine(full_model, test_loader, criterion, mode="Final Test")

print(f"\n🎉 KẾT QUẢ CUỐI CÙNG TRÊN TẬP TEST SAU 5 EPOCHS: {test_acc:.2%}")


🔓 MODE: JOINT TRAINING (Mở khóa LoRA)
👉 Ready for Phase 2.

🔥 BẮT ĐẦU HUẤN LUYỆN BỔ SUNG 2 EPOCHS (JOINT TRAINING)...

--- Epoch 1/2 ---


Training Epoch 1:   0%|          | 0/932 [00:00<?, ?it/s]

Evaluating (Val (Epoch 1)):   0%|          | 0/117 [00:00<?, ?it/s]

   📊 Val (Epoch 1) Result: Loss = 0.0700 | Accuracy = 0.9807
🌟 Đã lưu mô hình có Val Acc tốt nhất: 0.9807

--- Epoch 2/2 ---


Training Epoch 2:   0%|          | 0/932 [00:00<?, ?it/s]

Evaluating (Val (Epoch 2)):   0%|          | 0/117 [00:00<?, ?it/s]

   📊 Val (Epoch 2) Result: Loss = 0.1623 | Accuracy = 0.9716

--- Đánh giá cuối cùng với mô hình tốt nhất vừa lưu ---


Evaluating (Final Test):   0%|          | 0/117 [00:00<?, ?it/s]

   📊 Final Test Result: Loss = 0.0573 | Accuracy = 0.9828

🎉 KẾT QUẢ CUỐI CÙNG TRÊN TẬP TEST SAU 5 EPOCHS: 98.28%


In [ ]:
# -----------------------------------------------------------------------------
# 5. TIẾP TỤC HUẤN LUYỆN BỔ SUNG (5 EPOCHS)
# -----------------------------------------------------------------------------
num_epochs = 5
unfreeze_lora(full_model)
optimizer_warmup = optim.AdamW(filter(lambda p: p.requires_grad, full_model.parameters()), lr=3e-4)
print(f"\n🔥 BẮT ĐẦU HUẤN LUYỆN BỔ SUNG {num_epochs} EPOCHS (JOINT TRAINING)...")

for epoch in range(num_epochs):
    print(f"\n--- Epoch {epoch + 1}/{num_epochs} ---")

    # 1. Chạy huấn luyện trên tập Train
    # Sử dụng optimizer_joint để cập nhật cả LoRA và các lớp GNN
    train_loss, train_acc = train_engine(
        full_model,
        train_loader,
        optimizer_joint,
        criterion,
        f"Epoch {epoch + 1}"
    )

    # 2. Đánh giá trên tập Validation sau mỗi epoch
    val_loss, val_acc = evaluate_engine(
        full_model,
        val_loader,
        criterion,
        mode=f"Val (Epoch {epoch + 1})"
    )

    # 3. Cơ chế Lưu mô hình tốt nhất (Early Saving)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(full_model.state_dict(), "best_phishing_model.pt")
        print(f"🌟 Đã lưu mô hình có Val Acc tốt nhất: {val_acc:.4f}")

# -----------------------------------------------------------------------------
# 6. ĐÁNH GIÁ CUỐI CÙNG SAU KHI TRAIN XONG
# -----------------------------------------------------------------------------
print("\n--- Đánh giá cuối cùng với mô hình tốt nhất vừa lưu ---")
# Tải lại trọng số tốt nhất trước khi test
full_model.load_state_dict(torch.load("best_phishing_model.pt"))

test_loss, test_acc = evaluate_engine(full_model, test_loader, criterion, mode="Final Test")

print(f"\n🎉 KẾT QUẢ CUỐI CÙNG TRÊN TẬP TEST SAU 5 EPOCHS: {test_acc:.2%}")


🔓 MODE: JOINT TRAINING (Mở khóa LoRA)
👉 Ready for Phase 2.

🔥 BẮT ĐẦU HUẤN LUYỆN BỔ SUNG 5 EPOCHS (JOINT TRAINING)...

--- Epoch 1/5 ---


Training Epoch 1:   0%|          | 0/932 [00:00<?, ?it/s]

Evaluating (Val (Epoch 1)):   0%|          | 0/117 [00:00<?, ?it/s]

   📊 Val (Epoch 1) Result: Loss = 0.0873 | Accuracy = 0.9775

--- Epoch 2/5 ---


Training Epoch 2:   0%|          | 0/932 [00:00<?, ?it/s]

...

Training Epoch 5:   0%|          | 0/932 [00:00<?, ?it/s]

Evaluating (Val (Epoch 5)):   0%|          | 0/117 [00:00<?, ?it/s]

   📊 Val (Epoch 5) Result: Loss = 0.0766 | Accuracy = 0.9817

--- Đánh giá cuối cùng với mô hình tốt nhất vừa lưu ---


Evaluating (Final Test):   0%|          | 0/117 [00:00<?, ?it/s]

   📊 Final Test Result: Loss = 0.0724 | Accuracy = 0.9855

🎉 KẾT QUẢ CUỐI CÙNG TRÊN TẬP TEST SAU 5 EPOCHS: 98.55%
